# Polymarket Calibration Analysis

Reads `services/polymarket/data/calibration.db` (written by `scripts/calibrate_polymarket.sh` / `calibrate.py`) and scores Claude's probability estimates against resolved markets.

**Decision criteria** (baked into the plan — do not move the goalposts after seeing results):

| Outcome | Criteria |
|---|---|
| **Green** — tighten scanner, keep simulation running | Claude Brier < market Brier by ≥ 0.01 on N ≥ 200, AND edge-conditional accuracy (\|edge\| ≥ 5%) ≥ 58%, AND reliability roughly diagonal (no decile > 10pt off) |
| **Yellow** — iterate prompt/threshold | Any single criterion fails, others pass |
| **Red** — kill this prompt/model | Claude Brier ≥ market Brier OR edge-conditional accuracy ≤ 52% |

Run locally (not in the container) so the Jupyter stack stays out of the Pi image.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DB = Path('../services/polymarket/data/calibration.db')
assert DB.exists(), f'No calibration DB at {DB}. Run scripts/calibrate_polymarket.sh first.'

conn = sqlite3.connect(DB)
df = pd.read_sql('SELECT * FROM calibration_runs', conn)
conn.close()
print(f'rows: {len(df)}, runs: {df.run_id.nunique()}, models: {df.claude_model.unique().tolist()}')
df.head()

## 1. Brier scores — Claude vs. market

Brier = mean((predicted - outcome)²). Lower is better. Claude must beat the market's midpoint by at least 0.01 on ≥ 200 markets to meet the green criterion.

In [ ]:
def brier(pred, outcome):
    return float(np.mean((np.array(pred) - np.array(outcome)) ** 2))

def bootstrap_ci(pred, outcome, n=1000, alpha=0.05, fn=brier):
    rng = np.random.default_rng(42)
    pred = np.array(pred); outcome = np.array(outcome)
    samples = np.array([
        fn(pred[idx], outcome[idx])
        for idx in (rng.integers(0, len(pred), len(pred)) for _ in range(n))
    ])
    return float(np.quantile(samples, alpha/2)), float(np.quantile(samples, 1 - alpha/2))

b_claude = brier(df.claude_prob, df.resolution)
b_market = brier(df.market_prob_at_T, df.resolution)
ci_claude = bootstrap_ci(df.claude_prob.values, df.resolution.values)
ci_market = bootstrap_ci(df.market_prob_at_T.values, df.resolution.values)
print(f'N = {len(df)}')
print(f'Claude Brier  = {b_claude:.4f}  (95% CI: {ci_claude[0]:.4f} — {ci_claude[1]:.4f})')
print(f'Market Brier  = {b_market:.4f}  (95% CI: {ci_market[0]:.4f} — {ci_market[1]:.4f})')
print(f'Delta (market - claude) = {b_market - b_claude:+.4f}   (>= +0.01 is green)')

## 2. Log-loss

Penalizes overconfident wrong predictions harder than Brier. Claude should also win here if it's genuinely calibrated.

In [ ]:
def logloss(pred, outcome, eps=1e-6):
    p = np.clip(np.array(pred), eps, 1 - eps)
    y = np.array(outcome)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

print(f'Claude log-loss = {logloss(df.claude_prob, df.resolution):.4f}')
print(f'Market log-loss = {logloss(df.market_prob_at_T, df.resolution):.4f}')

## 3. Reliability diagram

Bin predictions by decile. For each bin, plot mean predicted probability (x) vs. realized frequency (y). The closer to the diagonal, the better calibrated. Systematic deviations > 10pt in any decile are a red flag.

In [ ]:
def reliability_bins(pred, outcome, n_bins=10):
    pred = np.array(pred); outcome = np.array(outcome)
    edges = np.linspace(0, 1, n_bins + 1)
    out = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (pred >= lo) & (pred < hi) if hi < 1 else (pred >= lo) & (pred <= hi)
        if mask.sum() == 0:
            continue
        out.append({'bin_lo': lo, 'bin_hi': hi, 'n': int(mask.sum()),
                    'mean_pred': float(pred[mask].mean()),
                    'realized': float(outcome[mask].mean())})
    return pd.DataFrame(out)

claude_rel = reliability_bins(df.claude_prob, df.resolution)
market_rel = reliability_bins(df.market_prob_at_T, df.resolution)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='perfect calibration')
ax.scatter(claude_rel.mean_pred, claude_rel.realized, s=claude_rel.n * 2,
           alpha=0.7, label='Claude', color='#2a7')
ax.scatter(market_rel.mean_pred, market_rel.realized, s=market_rel.n * 2,
           alpha=0.7, label='Market', color='#c34')
ax.set_xlabel('Predicted P(YES)'); ax.set_ylabel('Realized P(YES)')
ax.set_title('Reliability diagram (dot size ∝ bin count)')
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print('\nClaude deviations per bin:')
print(claude_rel.assign(dev=lambda d: d.realized - d.mean_pred)[['bin_lo', 'bin_hi', 'n', 'mean_pred', 'realized', 'dev']])

## 4. Edge-conditional accuracy

The strategy only acts on markets where `|claude_prob - market_prob| >= 5%`. So the only calibration number that matters for the PnL story is: *in that subset, is Claude right more than half the time?* Green bar is ≥ 58%.

In [ ]:
df = df.copy()
df['edge'] = (df.claude_prob - df.market_prob_at_T).abs()
df['claude_side'] = np.where(df.claude_prob >= df.market_prob_at_T, 1, 0)
df['correct'] = (df.claude_side == df.resolution).astype(int)

buckets = [0.0, 0.05, 0.10, 0.15, 0.25, 1.0]
df['edge_bucket'] = pd.cut(df.edge, buckets, include_lowest=True)
table = df.groupby('edge_bucket', observed=True).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    mean_edge=('edge', 'mean'),
)
print(table)
print(f'\nSubset |edge| >= 5%:  N = {(df.edge >= 0.05).sum()}, accuracy = {df.loc[df.edge >= 0.05, "correct"].mean():.3f}')
print(f'Green threshold:      0.58')

## 5. Confidence calibration

Does `claude_confidence` actually predict accuracy? If Claude says 0.9 confidence but is right 55% of the time, the confidence field is noise and we should drop the `min_confidence` filter.

In [ ]:
conf_buckets = [0.0, 0.5, 0.7, 0.85, 1.01]
df['conf_bucket'] = pd.cut(df.claude_confidence, conf_buckets, include_lowest=True)
print(df.groupby('conf_bucket', observed=True).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean'),
    mean_conf=('claude_confidence', 'mean'),
))

## 6. Category breakdown

Per the article's failure-mode analysis, sports should be noisiest. Crypto/macro may show real edge if any exists.

In [ ]:
if 'category' in df.columns and df.category.notna().any():
    cat_stats = df.groupby(df.category.fillna('unknown')).apply(
        lambda g: pd.Series({
            'n': len(g),
            'claude_brier': brier(g.claude_prob, g.resolution),
            'market_brier': brier(g.market_prob_at_T, g.resolution),
            'delta': brier(g.market_prob_at_T, g.resolution) - brier(g.claude_prob, g.resolution),
            'edge_acc': float((g.correct[g.edge >= 0.05]).mean()) if (g.edge >= 0.05).any() else float('nan'),
        }),
        include_groups=False,
    ).sort_values('delta', ascending=False)
    print(cat_stats)
else:
    print('No category labels present — CLOB did not surface category on these rows.')

## 7. Decision

Copy the output of this cell into `docs/polymarket_calibration_YYYY-MM-DD.md` along with a screenshot of the reliability diagram.

In [ ]:
N = len(df)
edge_mask = df.edge >= 0.05
edge_n = int(edge_mask.sum())
edge_acc = float(df.loc[edge_mask, 'correct'].mean()) if edge_n else float('nan')
delta = b_market - b_claude
max_dev = float(claude_rel.assign(dev=lambda d: (d.realized - d.mean_pred).abs()).dev.max()) if len(claude_rel) else float('nan')

green = (N >= 200 and delta >= 0.01 and edge_n > 0 and edge_acc >= 0.58 and max_dev <= 0.10)
red = (delta <= 0 or (edge_n > 0 and edge_acc <= 0.52))
verdict = 'GREEN' if green else ('RED' if red else 'YELLOW')

print(f'N = {N}')
print(f'Brier delta (market - claude) = {delta:+.4f}  (>= +0.01 for green)')
print(f'Edge subset (|edge| >= 5%):  N = {edge_n}, accuracy = {edge_acc:.3f}  (>= 0.58 for green)')
print(f'Max reliability deviation in any decile = {max_dev:.3f}  (<= 0.10 for green)')
print(f'\n=> VERDICT: {verdict}')